# Proyecto de Minería de Datos - Tercer corte
## Predicción de enfermedad cardiovascular (Dataset Heart Disease, Cleveland - UCI)

**Integrantes:** Héctor Cervantes Echeverria, Juan Guillermo Sarmiento Garrido, Santiago Valle Palacio y Diego Mendoza Gutiérrez.

Este notebook es la versión **actualizada** del modelado del segundo corte. Es el que genera los archivos que usa la aplicación de Streamlit (`modelo_pipeline.pkl`, `metricas.json`, `ejemplos_test.json` y las figuras).

### Qué cambió respecto al segundo corte

| Aspecto | Segundo corte | Tercer corte |
|---|---|---|
| GridSearch | Quedó interrumpido y no produjo un modelo final | Se completa con una rejilla moderada (96 combinaciones x 5 folds) |
| Selección del modelo final | Sin criterio explícito | Se elige solo con validación cruzada sobre el entrenamiento, sin mirar el conjunto de prueba |
| Validación cruzada | No se reportaba | Se reporta media, desviación y puntaje de entrenamiento (para detectar sobreajuste) |
| Guardado del modelo | No se guardaba | Pipeline completo guardado con `joblib` |
| Tipos de datos | Mezcla de enteros y flotantes | Todas las variables como `float`, igual que en la aplicación |
| Verificación | No existía | Se recarga el `.pkl` y se comprueba que reproduce las predicciones de registros del conjunto de prueba |

**Aviso:** proyecto académico. Los resultados no constituyen un diagnóstico médico.

## 0. Configuración

In [ ]:
import json
import shutil
import warnings
from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import sklearn
from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
    roc_curve,
)
from sklearn.model_selection import (
    GridSearchCV,
    StratifiedKFold,
    cross_val_score,
    cross_validate,
    train_test_split,
)
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid", palette="muted")

SEMILLA = 42
URL_UCI = (
    "https://archive.ics.uci.edu/ml/machine-learning-databases/"
    "heart-disease/processed.cleveland.data"
)
COLUMNAS = [
    "age", "sex", "cp", "trestbps", "chol", "fbs", "restecg",
    "thalach", "exang", "oldpeak", "slope", "ca", "thal", "target",
]
VAR_CONTINUAS = ["age", "trestbps", "chol", "thalach", "oldpeak"]
VAR_CATEGORICAS = ["sex", "cp", "fbs", "restecg", "exang", "slope", "ca", "thal"]
COLUMNAS_X = VAR_CONTINUAS + VAR_CATEGORICAS

NOMBRES = {
    "age": "Edad",
    "sex": "Sexo",
    "cp": "Tipo de dolor toracico",
    "trestbps": "Presion arterial en reposo",
    "chol": "Colesterol serico",
    "fbs": "Glucemia en ayunas > 120",
    "restecg": "ECG en reposo",
    "thalach": "Frecuencia cardiaca maxima",
    "exang": "Angina por ejercicio",
    "oldpeak": "Depresion del ST",
    "slope": "Pendiente del ST",
    "ca": "Vasos coloreados",
    "thal": "Talasemia (talio)",
}

SALIDA = Path("salida_app")
SALIDA.mkdir(exist_ok=True)


def crear_pipeline(clasificador):
    """Preprocesamiento + modelo en un solo objeto (evita fuga de informacion)."""
    preprocesador = ColumnTransformer(
        transformers=[
            ("num", StandardScaler(), VAR_CONTINUAS),
            ("cat", OneHotEncoder(drop="first", handle_unknown="ignore"), VAR_CATEGORICAS),
        ]
    )
    return Pipeline(steps=[("preprocesador", preprocesador), ("clasificador", clasificador)])


print("Version de scikit-learn:", sklearn.__version__)

: 

## 1. Carga, limpieza y partición de los datos

**Fuente:** UCI Machine Learning Repository, Heart Disease (subconjunto de Cleveland). https://archive.ics.uci.edu/dataset/45/heart+disease

- Los valores faltantes vienen marcados con `?`; se convierten a `NaN` y se eliminan (6 registros).
- La variable objetivo original (0 a 4) se binariza: 0 = sin enfermedad, 1 = con enfermedad.
- Se codifican de forma real, tal como vienen en los datos: `cp` de 1 a 4, `slope` de 1 a 3 y `thal` con los valores 3, 6 y 7.
- Todas las variables se convierten a `float` para que el modelo reciba el mismo tipo de dato que la aplicación.
- La partición es estratificada: 80 % entrenamiento y 20 % prueba.

In [ ]:
df = pd.read_csv(URL_UCI, names=COLUMNAS, na_values="?")
df["target"] = (df["target"] > 0).astype(int)

n_original = len(df)
df_limpio = df.drop_duplicates().dropna().copy()
print(f"Registros originales: {n_original} | Registros limpios: {len(df_limpio)}")

X = df_limpio[COLUMNAS_X].astype(float)
y = df_limpio["target"].astype(int)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=SEMILLA, stratify=y
)
print(f"Entrenamiento: {len(X_train)} | Prueba: {len(X_test)}")
print("\nDistribucion del objetivo (0 = sin enfermedad, 1 = con enfermedad):")
print(y.value_counts().rename({0: "Sin enfermedad", 1: "Con enfermedad"}))

## 2. Análisis exploratorio (resumen)

Cuatro visualizaciones relevantes: balance de la variable objetivo, relación entre edad y frecuencia cardíaca máxima, tipo de dolor torácico según diagnóstico y matriz de correlación. La figura se guarda como `eda_figura.png` para la aplicación.

In [ ]:
df_plot = df_limpio.copy()
df_plot["cp"] = df_plot["cp"].astype(int)
df_plot["Diagnostico"] = df_plot["target"].map({0: "Sin enfermedad", 1: "Con enfermedad"})
paleta = {"Sin enfermedad": "#3498db", "Con enfermedad": "#e74c3c"}

fig, axes = plt.subplots(2, 2, figsize=(15, 10))

sns.countplot(
    ax=axes[0, 0], data=df_plot, x="Diagnostico", hue="Diagnostico",
    order=["Sin enfermedad", "Con enfermedad"], palette=paleta, legend=False,
)
axes[0, 0].set_title("Distribucion de la variable objetivo")
axes[0, 0].set_xlabel("")
axes[0, 0].set_ylabel("Numero de pacientes")

sns.scatterplot(
    ax=axes[0, 1], data=df_plot, x="age", y="thalach", hue="Diagnostico",
    palette=paleta, alpha=0.85,
)
axes[0, 1].set_title("Edad vs frecuencia cardiaca maxima")
axes[0, 1].set_xlabel("Edad (anios)")
axes[0, 1].set_ylabel("Frecuencia cardiaca maxima (lpm)")

sns.countplot(
    ax=axes[1, 0], data=df_plot, x="cp", hue="Diagnostico",
    hue_order=["Sin enfermedad", "Con enfermedad"], palette=paleta,
)
axes[1, 0].set_title("Tipo de dolor toracico (cp) segun diagnostico")
axes[1, 0].set_xlabel("Tipo de dolor toracico (1 = tipica, 4 = asintomatico)")
axes[1, 0].set_ylabel("Numero de pacientes")

corr = df_limpio.corr()
mascara = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(
    ax=axes[1, 1], data=corr, mask=mascara, annot=True, fmt=".2f",
    cmap="vlag", vmin=-1, vmax=1, cbar=False,
)
axes[1, 1].set_title("Matriz de correlacion")

plt.tight_layout()
plt.savefig(SALIDA / "eda_figura.png", dpi=150, bbox_inches="tight")
plt.show()

**Lectura de la figura:** el conjunto está razonablemente balanceado (160 sin enfermedad y 137 con enfermedad). Las correlaciones más altas con el objetivo son `thal`, `ca`, `exang`, `oldpeak`, `cp` y `thalach` (esta última negativa). `fbs` y `chol` casi no se relacionan linealmente con el objetivo.

## 3. Comparación de modelos base (igual que en el segundo corte)

Se entrenan seis algoritmos con el mismo pipeline de preprocesamiento. Además de las métricas en prueba se agrega el **recall en validación cruzada** (5 folds sobre el entrenamiento), que es una medida más estable que un conjunto de prueba de solo 60 pacientes.

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEMILLA)

modelos = {
    "Regresion Logistica": LogisticRegression(random_state=SEMILLA),
    "KNN": KNeighborsClassifier(),
    "Arbol de Decision": DecisionTreeClassifier(random_state=SEMILLA),
    "Random Forest": RandomForestClassifier(random_state=SEMILLA),
    "Gradient Boosting": GradientBoostingClassifier(random_state=SEMILLA),
    "SVM (RBF)": SVC(probability=True, random_state=SEMILLA),
}

resultados = []
for nombre, modelo in modelos.items():
    pipe = crear_pipeline(modelo)
    recall_cv = cross_val_score(pipe, X_train, y_train, cv=cv, scoring="recall").mean()
    pipe.fit(X_train, y_train)
    pred = pipe.predict(X_test)
    proba = pipe.predict_proba(X_test)[:, 1]
    resultados.append({
        "Modelo": nombre,
        "Accuracy": accuracy_score(y_test, pred),
        "Precision": precision_score(y_test, pred),
        "Recall": recall_score(y_test, pred),
        "F1-Score": f1_score(y_test, pred),
        "ROC-AUC": roc_auc_score(y_test, proba),
        "Recall CV (5 folds)": recall_cv,
    })

df_resultados = pd.DataFrame(resultados).sort_values(by="Recall CV (5 folds)", ascending=False)
print("--- COMPARACION DE MODELOS (metricas en prueba; ultima columna en validacion cruzada) ---")
print(df_resultados.round(3).to_string(index=False))

Se continúa con **Random Forest**, que es el algoritmo con el que se construyó la aplicación en el segundo corte. En un contexto médico se prioriza el **recall** (sensibilidad) porque un falso negativo, un paciente enfermo clasificado como sano, es el error más costoso.

## 4. Optimización de hiperparámetros y selección del modelo final

Se compara Random Forest con parámetros por defecto contra Random Forest optimizado con `GridSearchCV` (optimiza recall). **El modelo final se elige únicamente por el recall en validación cruzada del conjunto de entrenamiento**, nunca por el resultado en prueba, para no filtrar información del conjunto de prueba a la decisión.

In [ ]:
# 4.1 Random Forest por defecto
pipe_base = crear_pipeline(RandomForestClassifier(random_state=SEMILLA))
recall_cv_base = cross_val_score(pipe_base, X_train, y_train, cv=cv, scoring="recall").mean()
pipe_base.fit(X_train, y_train)

# 4.2 Random Forest optimizado
param_grid = {
    "clasificador__n_estimators": [100, 200],
    "clasificador__max_depth": [None, 3, 5, 8],
    "clasificador__min_samples_split": [2, 5, 10],
    "clasificador__criterion": ["gini", "entropy"],
}
print("Ejecutando GridSearchCV (96 combinaciones x 5 folds)...")
grid = GridSearchCV(
    crear_pipeline(RandomForestClassifier(random_state=SEMILLA)),
    param_grid, cv=cv, scoring="recall", n_jobs=-1,
)
grid.fit(X_train, y_train)
recall_cv_opt = grid.best_score_

print("Mejores parametros:", grid.best_params_)
print(f"Recall CV (por defecto) : {recall_cv_base:.3f}")
print(f"Recall CV (optimizado)  : {recall_cv_opt:.3f}")

# 4.3 Seleccion del modelo final solo con validacion cruzada
if recall_cv_opt > recall_cv_base:
    modelo_final = grid.best_estimator_
    nombre_modelo = "Random Forest optimizado (GridSearchCV)"
    mejores_params = {k.replace("clasificador__", ""): v for k, v in grid.best_params_.items()}
else:
    modelo_final = pipe_base
    nombre_modelo = "Random Forest (parametros por defecto)"
    mejores_params = {"n_estimators": 100, "criterion": "gini", "max_depth": None,
                      "min_samples_split": 2}
print("\nModelo final seleccionado:", nombre_modelo)

## 5. Evaluación del modelo final

Se reportan las métricas en el conjunto de prueba, la matriz de confusión, el reporte por clase y la validación cruzada (con el puntaje de entrenamiento al lado, para detectar sobreajuste).

In [ ]:
y_pred = modelo_final.predict(X_test)
y_proba = modelo_final.predict_proba(X_test)[:, 1]

metricas_test = {
    "accuracy": accuracy_score(y_test, y_pred),
    "precision": precision_score(y_test, y_pred),
    "recall": recall_score(y_test, y_pred),
    "f1": f1_score(y_test, y_pred),
    "roc_auc": roc_auc_score(y_test, y_proba),
}
tn, fp, fn, tp = confusion_matrix(y_test, y_pred, labels=[0, 1]).ravel()
especificidad = tn / (tn + fp)

print("--- METRICAS EN PRUEBA ---")
for k, v in metricas_test.items():
    print(f"{k:10s}: {v:.3f}")
print(f"especificidad: {especificidad:.3f}")
print(f"Matriz de confusion -> VN={tn} FP={fp} FN={fn} VP={tp}")
print("\nReporte por clase:")
print(classification_report(y_test, y_pred, target_names=["Sin enfermedad", "Con enfermedad"]))

cv_res = cross_validate(
    clone(modelo_final), X_train, y_train, cv=cv, n_jobs=-1, return_train_score=True,
    scoring={"accuracy": "accuracy", "precision": "precision", "recall": "recall",
             "f1": "f1", "roc_auc": "roc_auc"},
)
metricas_cv = {}
print("--- VALIDACION CRUZADA (5 folds, solo entrenamiento) ---")
for m in ["accuracy", "precision", "recall", "f1", "roc_auc"]:
    val = cv_res[f"test_{m}"]
    tr = cv_res[f"train_{m}"]
    metricas_cv[m] = {
        "val_media": float(val.mean()),
        "val_std": float(val.std()),
        "train_media": float(tr.mean()),
    }
    print(f"{m:10s}: validacion {val.mean():.3f} +/- {val.std():.3f} | entrenamiento {tr.mean():.3f}")

### Figuras de evaluación

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
cm = confusion_matrix(y_test, y_pred, labels=[0, 1])
ConfusionMatrixDisplay(cm, display_labels=["Sin enfermedad", "Con enfermedad"]).plot(
    ax=axes[0], cmap="Blues", values_format="d", colorbar=False
)
axes[0].set_title(f"Matriz de confusion (prueba, n={len(y_test)})")
axes[0].grid(False)

fpr, tpr, _ = roc_curve(y_test, y_proba)
axes[1].plot(fpr, tpr, color="darkorange", lw=2,
             label=f"Curva ROC (AUC = {metricas_test['roc_auc']:.3f})")
axes[1].plot([0, 1], [0, 1], color="navy", lw=2, linestyle="--", label="Azar")
axes[1].set_xlabel("Tasa de falsos positivos (1 - especificidad)")
axes[1].set_ylabel("Tasa de verdaderos positivos (recall)")
axes[1].set_title("Curva ROC")
axes[1].legend(loc="lower right")
plt.tight_layout()
plt.savefig(SALIDA / "matriz_confusion_roc.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# Importancia por variable original (suma de las columnas one-hot de cada una)
prep = modelo_final.named_steps["preprocesador"]
nombres_salida = prep.get_feature_names_out()
importancias = modelo_final.named_steps["clasificador"].feature_importances_

acumulado = {}
for nombre, imp in zip(nombres_salida, importancias):
    base = nombre.split("__", 1)[1].split("_", 1)[0]
    acumulado[base] = acumulado.get(base, 0.0) + float(imp)

df_imp = (
    pd.DataFrame({"codigo": list(acumulado.keys()), "importancia": list(acumulado.values())})
    .assign(variable=lambda d: d["codigo"].map(NOMBRES))
    .sort_values("importancia", ascending=False)
    .reset_index(drop=True)
)
print(df_imp[["variable", "importancia"]].round(3).to_string(index=False))

top10 = df_imp.head(10)
plt.figure(figsize=(10, 6))
sns.barplot(data=top10, x="importancia", y="variable", hue="variable",
            palette="viridis", legend=False)
plt.title("Top 10 variables clinicas por importancia (Random Forest)")
plt.xlabel("Importancia relativa (suma de las categorias de cada variable)")
plt.ylabel("")
plt.tight_layout()
plt.savefig(SALIDA / "importancia_variables.png", dpi=150, bbox_inches="tight")
plt.show()

## 6. Guardado del pipeline y de los archivos de la aplicación

Se guarda el **pipeline completo** (preprocesamiento + modelo) con `joblib`, de modo que la aplicación no entrene nada al ejecutarse. También se guardan las métricas reales, tres registros conocidos del conjunto de prueba (uno sano, uno enfermo y, si existe, uno mal clasificado) y el `requirements.txt` con la versión exacta de scikit-learn.

In [ ]:
# Registros conocidos del conjunto de prueba
indices = [y_test[y_test == 0].index[0], y_test[y_test == 1].index[0]]
errados = X_test.index[(y_pred != y_test.to_numpy())]
if len(errados) > 0 and errados[0] not in indices:
    indices.append(errados[0])

ejemplos = []
for idx in indices:
    fila = X_test.loc[[idx]]
    pred = int(modelo_final.predict(fila)[0])
    prob = float(modelo_final.predict_proba(fila)[0, 1])
    ejemplos.append({
        "id": int(idx),
        "descripcion": f"Registro {int(idx)} del conjunto de prueba",
        "entradas": {c: float(fila.iloc[0][c]) for c in COLUMNAS_X},
        "real": int(y_test.loc[idx]),
        "prediccion_notebook": pred,
        "probabilidad_notebook": prob,
    })
    print(f"Registro {idx}: real={int(y_test.loc[idx])} prediccion={pred} probabilidad={prob:.4f}")

# Pipeline
joblib.dump(modelo_final, SALIDA / "modelo_pipeline.pkl", compress=3)

# Metricas
rangos = {c: [float(df_limpio[c].min()), float(df_limpio[c].max())] for c in VAR_CONTINUAS}
metricas = {
    "modelo": nombre_modelo,
    "criterio_seleccion": "Mayor recall medio en validacion cruzada de 5 folds (solo entrenamiento)",
    "recall_cv_base": float(recall_cv_base),
    "recall_cv_optimizado": float(recall_cv_opt),
    "mejores_parametros": mejores_params,
    "sklearn_version": sklearn.__version__,
    "n_original": int(n_original),
    "n_limpio": int(len(df_limpio)),
    "n_train": int(len(X_train)),
    "n_test": int(len(X_test)),
    "test": {k: float(v) for k, v in metricas_test.items()},
    "especificidad": float(especificidad),
    "matriz_confusion": {"tn": int(tn), "fp": int(fp), "fn": int(fn), "tp": int(tp)},
    "cv": metricas_cv,
    "importancia_variables": [
        {"variable": r.variable, "importancia": float(r.importancia)} for r in df_imp.itertuples()
    ],
    "rangos_entrenamiento": rangos,
}
with open(SALIDA / "metricas.json", "w", encoding="utf-8") as f:
    json.dump(metricas, f, ensure_ascii=False, indent=2)
with open(SALIDA / "ejemplos_test.json", "w", encoding="utf-8") as f:
    json.dump(ejemplos, f, ensure_ascii=False, indent=2)

# requirements.txt
with open(SALIDA / "requirements.txt", "w", encoding="utf-8") as f:
    f.write(
        "streamlit>=1.40\n"
        "pandas\n"
        "numpy\n"
        f"scikit-learn=={sklearn.__version__}\n"
        "joblib\n"
    )

# Tabla para el README
tabla = (
    f"Modelo final: {nombre_modelo}\n\n"
    f"| Metrica | Prueba (n={len(X_test)}) | Validacion cruzada (media +/- desv.) |\n"
    "|---|---|---|\n"
)
etiquetas = {"accuracy": "Exactitud", "precision": "Precision", "recall": "Recall",
             "f1": "F1-score", "roc_auc": "ROC-AUC"}
for m, etiqueta in etiquetas.items():
    tabla += (
        f"| {etiqueta} | {metricas_test[m]:.3f} | "
        f"{metricas_cv[m]['val_media']:.3f} +/- {metricas_cv[m]['val_std']:.3f} |\n"
    )
tabla += (
    f"\nMatriz de confusion en prueba: VN={tn}, FP={fp}, FN={fn}, VP={tp}. "
    f"Especificidad: {especificidad:.3f}. Mejores parametros: {mejores_params}\n"
)
(SALIDA / "resultados.md").write_text(tabla, encoding="utf-8")
print("\n" + tabla)

print("Archivos generados en", SALIDA.resolve())
for p in sorted(SALIDA.iterdir()):
    print("  -", p.name)

## 7. Verificación: el modelo guardado reproduce las predicciones

Se recarga el archivo `modelo_pipeline.pkl` desde disco, como lo hace la aplicación, y se comprueba que las predicciones y probabilidades de los registros de prueba coinciden con las calculadas arriba. Esta es la misma comparación que hace la aplicación en la sección "Verificación con un registro conocido del conjunto de prueba".

In [ ]:
modelo_recargado = joblib.load(SALIDA / "modelo_pipeline.pkl")

for e in ejemplos:
    entrada = pd.DataFrame([e["entradas"]], columns=COLUMNAS_X).astype(float)
    pred = int(modelo_recargado.predict(entrada)[0])
    prob = float(modelo_recargado.predict_proba(entrada)[0, 1])
    coincide = pred == e["prediccion_notebook"] and abs(prob - e["probabilidad_notebook"]) < 1e-6
    print(f"{e['descripcion']}: prediccion={pred} probabilidad={prob:.4f} -> "
          f"{'COINCIDE' if coincide else 'NO COINCIDE'}")
    assert coincide

# Prueba adicional: el pipeline recargado reproduce las predicciones de TODO el conjunto de prueba
assert (modelo_recargado.predict(X_test) == y_pred).all()
print("\nEl pipeline recargado reproduce las predicciones de los", len(X_test), "registros de prueba.")

## 8. Empaquetado para la aplicación

In [ ]:
shutil.make_archive("archivos_app", "zip", SALIDA)
print("Creado: archivos_app.zip")

try:
    from google.colab import files  # solo existe en Colab
    files.download("archivos_app.zip")
except ImportError:
    print("No se esta en Colab: tome 'archivos_app.zip' de la carpeta actual.")

## 9. Conclusiones y limitaciones

**Resultados.** El modelo final es un Random Forest elegido por recall en validación cruzada. Las métricas exactas están en las salidas de las secciones 4 y 5 y se resumen en el `README.md` del repositorio. La exactitud supera el umbral de revisión de 0,70, pero el análisis no se detiene ahí: se revisaron la matriz de confusión, el recall y la estabilidad de la validación cruzada.

**Comparación con el segundo corte.** El Random Forest por defecto y el optimizado quedan muy cerca en el conjunto de prueba; la diferencia en recall equivale a un solo paciente. La optimización de hiperparámetros no aporta una mejora clara en este conjunto de datos, y así se reporta.

**Limitaciones.**
- Solo hay 297 registros y el conjunto de prueba tiene 60 pacientes: cada paciente mal clasificado mueve las métricas varios puntos porcentuales. Por eso se reporta también la validación cruzada.
- Los datos provienen de una sola institución y de una época específica; el modelo puede no generalizar a otras poblaciones.
- Los falsos negativos (pacientes enfermos clasificados como sanos) son el error más costoso; el modelo no debe usarse para descartar una enfermedad.
- La variable objetivo se binariza y no distingue gravedad.
- Es un prototipo académico, no certificado para uso clínico.